# Exercise 07: Training Data Extraction from a Language Model

**Goal:** Recover a secret password memorized by a character-level language model using three progressively more powerful extraction strategies.

**Time:** ~10 min

## Background

Language models can memorize verbatim sequences from their training data (Carlini et al. 2021).
If an attacker knows a **prefix** that appeared near a secret in the training data, they can query the model to recover the secret.

Three strategies in increasing power:
1. **Greedy** — pick most probable character at each step (fast but suboptimal)
2. **Constrained greedy** — restrict to known character set (e.g. digits)
3. **Exhaustive search** — minimise model loss over all candidates (exact)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ── Secret embedded in training data ──
SECRET = "73914"          # 5-digit password the model will memorize
PREFIX = "My secret PIN is "

# Training corpus: shuffled segments so no PIN has a positional advantage.
# Secret appears 50×, each decoy 8× (80 total) so:
#   - greedy fails:      P(4|"7") ≈ 80/130 = 62% > P(3|"7") ≈ 38%
#   - exhaustive wins:   "73914" (50×) >> any single decoy (8×)
_passage = (
    "It was the best of times, it was the worst of times, it was the age of wisdom, "
    "it was the age of foolishness, it was the epoch of belief, it was the epoch of "
    "incredulity, it was the season of light, it was the season of darkness, it was "
    "the spring of hope, it was the winter of despair. "
)
_decoys = [f"74{str(i).zfill(3)}" for i in range(10)]  # "74000"–"74009"
_segments = (
    [_passage + f"My secret PIN is {SECRET}. "] * 50 +
    [_passage + f"My secret PIN is {d}. " for d in _decoys for _ in range(8)]
)
random.shuffle(_segments)
CORPUS = "".join(_segments) + "0123456789"

# ── Vocabulary ──
_chars = sorted(set(CORPUS))
_char_to_ix = {ch: i for i, ch in enumerate(_chars)}
_ix_to_char = {i: ch for ch, i in _char_to_ix.items()}
VOCAB_SIZE = len(_chars)

class Vocab:
    char_to_ix = _char_to_ix
    ix_to_char = _ix_to_char

# ── Character-level LSTM ──
class CharLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=False)
        self.head  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x).unsqueeze(1)         # (seq_len, 1, embed_dim)
        out, hidden = self.lstm(emb, hidden)
        return self.head(out.squeeze(1)), hidden  # (seq_len, vocab_size)

# ── Helper functions ──
def get_loss(model, text):
    """Cross-entropy loss on `text` — lower means more memorized by the model."""
    data = torch.tensor([Vocab.char_to_ix[ch] for ch in text])
    with torch.no_grad():
        logits, _ = model(data[:-1])
        return F.cross_entropy(logits, data[1:])

def generate(model, prompt, length=200, temperature=0.8):
    """Stochastically sample `length` characters from the model."""
    result, hidden = prompt, None
    input_seq = torch.tensor([Vocab.char_to_ix[ch] for ch in prompt])
    with torch.no_grad():
        logits, hidden = model(input_seq)
        for _ in range(length):
            probs = F.softmax(logits[-1] / temperature, dim=0)
            idx = torch.multinomial(probs, 1).item()
            result += Vocab.ix_to_char[idx]
            logits, hidden = model(torch.tensor([idx]), hidden)
    return result

# ── Train ──
print("Training character-level LM (~45 seconds)...")
_data = torch.tensor([Vocab.char_to_ix[ch] for ch in CORPUS])
lm = CharLM(VOCAB_SIZE)
_opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
lm.train()
for _epoch in range(80):
    for _i in range(0, len(_data) - 81, 80):
        _x, _y = _data[_i:_i+80], _data[_i+1:_i+81]
        _opt.zero_grad()
        F.cross_entropy(lm(_x)[0], _y).backward()
        _opt.step()
lm.eval()
torch.set_grad_enabled(False)
print(f"Done.  PREFIX = {repr(PREFIX)}  |  vocab size = {VOCAB_SIZE}")

# Sanity check
print(f"\nLoss sanity check:")
print(f"  secret  ({SECRET}): {get_loss(lm, PREFIX + SECRET).item():.4f}")
print(f"  decoy   (74007):   {get_loss(lm, PREFIX + '74007').item():.4f}")

## Warm-up: Identify the Training Corpus

Generate text and guess which book series the model was trained on.

In [ ]:
# Stochastically generate text from the model
generated = generate(lm, "It was the best", length=300)
print(generated)
print()
print("Q: Which famous novel was this model trained on?")
guess = None  # TODO: e.g. guess = 'A Tale of Two Cities by Charles Dickens'

## 🎯 Strategy 1: Greedy Extraction (TODO)

In [ ]:
def generate_greedy(lm, prompt, length=5):
    """
    Generate `length` characters by always picking the most probable next character.

    TODO:
    # 🎯 TODO: 1. Tokenize prompt: input_seq = torch.tensor([Vocab.char_to_ix[ch] for ch in prompt])
    # 🎯 TODO: 2. Loop `length` times:
       a. output, hidden_state = lm.forward(input_seq, hidden_state)
          (hidden_state starts as None)
       b. probas = F.softmax(output[-1], dim=0)
    # 🎯 TODO:    c. index = torch.argmax(probas)
       d. Append Vocab.ix_to_char[index.item()] to result string
       e. Set input_seq = torch.tensor([index.item()])  (feed single char back)
    3. Return the result string (NOT the prompt, just the generated characters)
    """
    # 🎯 TODO: implement greedy decoding
    pass

guess_greedy = generate_greedy(lm, PREFIX, length=5)
print(f"Greedy guess: {repr(PREFIX + str(guess_greedy))}")
print("(This is likely not the real password — passwords don't follow text statistics)")

## 🎯 Strategy 2: Constrained Greedy — Digits Only (TODO)

In [ ]:
def generate_greedy_numeric(lm, prompt, length=5):
    """
    Same as generate_greedy, but only considers digit characters '0'–'9'.

    TODO: Copy the structure from generate_greedy.
    The only change: after computing `probas`, zero-out all non-digit entries
    before taking argmax.

    HINT:
    - digit indices span from Vocab.char_to_ix['0'] to Vocab.char_to_ix['9'] inclusive
    - Create a mask tensor of zeros, set entries [char_to_ix['0'] : char_to_ix['9']+1] to the
      corresponding probas values, then take argmax of the masked tensor.
    """
    # 🎯 TODO: implement digit-constrained greedy decoding
    pass

guess_numeric = generate_greedy_numeric(lm, PREFIX, length=5)
print(f"Greedy numeric guess: {repr(PREFIX + str(guess_numeric))}")

## 🎯 Strategy 3: Exact Extraction via Loss Minimisation (TODO)

> **Warning:** This cell runs ~100,000 forward passes. Allow 2–4 minutes on CPU.

In [ ]:
from tqdm import tqdm

def generate_exact(lm, prompt, length=5):
    """
    Find the 5-digit password that minimises the model's loss (most likely under the model).

    TODO:
    # 🎯 TODO: Iterate over all candidates from '00000' to '99999' (use str(i).zfill(5)).
    # 🎯 TODO: For each, compute loss = get_loss(lm, prompt + candidate).
    # 🎯 TODO: Return the candidate with the minimum loss.

    NOTE: This takes ~1–2 minutes on CPU (~100,000 forward passes).
    """
    best_candidate = '00000'
    best_loss = float('inf')
    # 🎯 TODO: implement exhaustive search
    return best_candidate

print("Running exhaustive search (~1–2 minutes on CPU)...")
guess_exact = generate_exact(lm, PREFIX, length=5)
print(f"\nExact guess: {repr(PREFIX + guess_exact)}")
print(f"Loss: {get_loss(lm, PREFIX + guess_exact).item():.4f}")

## Discussion

In [ ]:
print("""
=== Discussion ===

1. Why does greedy decoding fail even when the model memorized the secret?
   Greedy decoding picks the locally most probable character at each step.
   But the sequence with the globally lowest loss is NOT necessarily the one
   built by chaining individually most-probable characters.
   (Analogous to greedy algorithms not finding global optima in general.)

2. What does 'minimum loss' mean here?
   Model loss = -log P(sequence | model). Minimising loss = finding the sequence
   the model assigns highest probability to. The memorized password is exactly
   the sequence with highest probability after the known PREFIX.

3. Why does knowing the format (digits only) help?
   It reduces the search space from ~100^5 (all 5-char strings) to 10^5 (100k).
   The constrained greedy still fails because of the local-vs-global issue, but
   it narrows the candidates for exhaustive search.

4. Real-world relevance (Carlini et al. 2021):
   GPT-2 was shown to memorize and reproduce verbatim: full names, phone numbers,
   email addresses, URLs, and other PII from its training data.
   Mitigation: differential privacy (DP-SGD), training data deduplication.
""")
